In [ ]:
import os
import warnings
import logging
from PIL import Image
import torch
from torchvision.utils import save_image
from transformers import AutoModelForImageClassification, ViTImageProcessor
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras.losses")

# Параметры:
input_folder = "isic_1000_balanced"
output_folder = "isic_1000_fgsm"
model_path = "melanoma_attacks/checkpoint-12670"
epsilon = 8 / 255
# Метки:
LABEL_MAP = {
    "AK": 0, "BCC": 1, "BKL": 2, "DF": 3,
    "MEL": 4, "NV": 5, "SCC": 6, "VASC": 7
}

# Загрузка модели и процессора:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForImageClassification.from_pretrained(model_path).to(device).eval()
processor = ViTImageProcessor.from_pretrained(model_path)

def fgsm_attack(image, label_id):
    inputs = processor(images=image, return_tensors="pt").to(device)
    pixel_values = inputs['pixel_values'].clone().detach().requires_grad_(True)
    labels = torch.tensor([label_id]).to(device)
    outputs = model(pixel_values, labels=labels)
    loss = outputs.loss
    loss.backward()
    perturbation = epsilon * pixel_values.grad.sign()
    adv_pixel_values = pixel_values + perturbation
    adv_pixel_values = torch.clamp(adv_pixel_values, 0, 1)
    return adv_pixel_values.detach()

def save_adversarial_image(tensor, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_image(tensor.squeeze(0).cpu(), output_path)

# Обработка всех изображений:
total = 0

for class_name in os.listdir(input_folder):
    class_input_path = os.path.join(input_folder, class_name)
    class_output_path = os.path.join(output_folder, class_name)
    os.makedirs(class_output_path, exist_ok=True)
    label_id = LABEL_MAP[class_name]
    for filename in os.listdir(class_input_path):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        input_path = os.path.join(class_input_path, filename)
        output_path = os.path.join(class_output_path, filename)
        try:
            image = Image.open(input_path).convert("RGB")
            adv_tensor = fgsm_attack(image, label_id)
            save_adversarial_image(adv_tensor, output_path)
            total += 1
            if total % 50 == 0:
                print(f" Обработано {total} изображений...")
        except Exception as e:
            print(f"Ошибка с {input_path}: {e}")

print(f"\nFGSM-атака завершена. Искажённые изображения сохранены в: {output_folder}")
